In [81]:
import numpy as np
import pandas as pd
import seaborn as sns

In [118]:
df=pd.read_csv('fifa_world_cup_2026_player_performance.csv')

In [129]:
df.shape

(54600, 75)

In [5]:
groups = df.T.groupby(list(df.T)).groups

found_duplicate =False
for group in groups.values():
    if len(group)>1:
        print(list(group))
        found_duplicate=True
if not found_duplicate:
    print("No duplicate columns")

No duplicate columns


In [119]:
X=df.drop("market_value_eur",axis=1)
y=df['market_value_eur']
num_col=X.select_dtypes(include=['int64','float64']).columns

In [120]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [86]:
#from sklearn.preprocessing import StandardScaler
#scaler=StandardScaler()
#X_train[num_col]=scaler.fit_transform(X_train[num_col])
#X_test[num_col]=scaler.transform(X_test[num_col])

In [121]:
from sklearn.feature_selection import VarianceThreshold
sel=VarianceThreshold(threshold=0.1)

In [122]:
sel.fit(X_train[num_col])

VarianceThreshold(threshold=0.1)

In [123]:
num_col[~sel.get_support()]

Index(['goals', 'assists', 'shots_on_target', 'expected_goals_xg',
       'expected_assists_xa', 'pass_accuracy', 'successful_crosses',
       'yellow_cards', 'red_cards', 'save_percentage', 'punches',
       'clean_sheet', 'penalty_saves', 'player_of_match_awards'],
      dtype='object')

In [124]:
columns=X_train[num_col].columns[sel.get_support()]

In [125]:
X_train=sel.transform(X_train[num_col])
X_test=sel.transform(X_test[num_col])

In [126]:
X_train=pd.DataFrame(X_train,columns=columns)
X_test=pd.DataFrame(X_test,columns=columns)

In [134]:
X_train.shape

(36582, 46)

In [146]:
corr_matrix=X_train.corr(numeric_only=True)
columns=corr_matrix.columns
columns_to_drop=[]
for i in range(len(columns)):
    for j in range(i+1,len(columns)):
        if abs(corr_matrix.loc[columns[i],columns[j]])>=0.9:
            columns_to_drop.append(columns[j])

In [147]:
columns_to_drop=set(columns_to_drop)

In [149]:
print(len(columns_to_drop))

9


In [151]:
X_train=X_train.drop(columns=columns_to_drop)
X_test=X_test.drop(columns=columns_to_drop)

In [152]:
X_train.shape

(36582, 37)

In [168]:
from sklearn.ensemble import RandomForestRegressor
model2 =RandomForestRegressor(n_estimators=200,random_state=42,max_depth=12)


In [170]:
model2.fit(X_train,y_train)

RandomForestRegressor(max_depth=12, n_estimators=200, random_state=42)

In [166]:
y_pred=model2.predict(X_test)

In [171]:
print("Train Score =",model2.score(X_train,y_train))
print("Test Score =",model2.score(X_test,y_test))

Train Score = 0.8772770603492053
Test Score = 0.7331320234755306
